<a href="https://colab.research.google.com/github/Bezud94/machine-learning-assignment-2/blob/main/ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ldmCVTPj4XBMpdRlx7r7")
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to rock-paper-scissors-14 in yolov8:: 100%|██████████| 14682/14682 [00:01<00:00, 8720.28it/s] 


In [2]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/112.6 GB disk)


In [7]:
!yolo val model=yolo26n.pt data=/content/rock-paper-scissors-14/data.yaml

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1440.4±526.4 MB/s, size: 31.2 KB)
val: Scanning /content/rock-paper-scissors-14/valid/labels... 576 images, 238 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 576/576 2.2Kit/s 0.3s
val: New cache created: /content/rock-paper-scissors-14/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.8it/s 6.2s
                   all        576        400      0.347     0.0743    0.00503     0.0026
                person        132        139     0.0395      0.223     0.0151    0.00779
               bicycle        121        141          1          0          0          0
                   car        116        120          0          0   7.47e-06   5.23e-06
Speed: 1.2ms preprocess, 4.1ms inference, 0.0ms l

In [24]:
from ultralytics import YOLO

# Load a model
model = YOLO("/content/runs/detect/train-4/weights/best.pt")

# Train the model
results = model.train(data="/content/rock-paper-scissors-14/data.yaml", epochs=30, imgsz=640, batch=16, cache=True)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rock-paper-scissors-14/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train-4/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-5, nbs=6

In [31]:
import csv
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-5/weights/best.pt")

video_path = "test.mp4"
output_video = "tracking.mp4"
output_csv = "trajectory.csv"

track_history = defaultdict(list)
trajectory_records = []
frame_idx = 0

In [34]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    "tracking_full.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(frame, persist=True, conf=0.25, verbose=False)
    annotated_frame = results[0].plot()

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes_xywh = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for (cx, cy, w, h), track_id in zip(boxes_xywh, track_ids):
            track = track_history[track_id]
            track.append((float(cx), float(cy)))

            if len(track) > total_frames:
                track.pop(0)

            if len(track) > 1:
                points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(annotated_frame, [points], False, (0, 255, 255), 3)

    out.write(annotated_frame)

cap.release()
out.release()